# 04 — Evaluation & Error Analysis

**Goal:** Load a saved model checkpoint, run predictions on held-out data,
compute quantitative metrics, and visually inspect where the model succeeds and fails.

---

## Why error analysis matters

A single accuracy or mIoU number tells you *how well* a model performs on average,
but not *why* it fails or *which* failure modes are most important to fix.

Common failure modes in Mars terrain segmentation:
- **Sand/soil confusion** — these classes look similar in colour and texture.
- **Missed small rocks** — low-resolution features are hard for shallow models.
- **Shadow regions** — dark areas may be misclassified as bedrock.
- **Boundary bleeding** — class boundaries are smeared, especially at coarse resolution.

> **Prerequisite:** Run `03_baseline_training.ipynb` first to save a checkpoint.

In [1]:
import csv
import hashlib
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data_paths import ARTIFACTS_DIR, FIGURES_DIR, MODELS_DIR, PREDICTIONS_DIR, PROJECT_ROOT, RAW_DATA_DIR, ensure_project_dirs
from src.dataset import AI4MarsDataset, load_pairs_from_manifest
from src.foundation import build_checkpoint_metadata, build_split_manifests, detect_dataset_root, write_dataset_manifest
from src.train_utils import get_device, load_checkpoint
from src.metrics import intersection_over_union, segmentation_confusion_matrix
from src.visualize import CLASS_NAMES, show_image, show_image_mask_overlay

ensure_project_dirs()


All project directories are ready.


In [2]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_tensor(t: torch.Tensor) -> str:
    h = hashlib.sha256()
    h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

## Step 1 — Rebuild the Same Model Architecture

The checkpoint only stores **weights**, not the model class definition.  
You must define the same architecture here before loading.

In [3]:
import segmentation_models_pytorch as smp

NUM_CLASSES = 4
IGNORE_INDEX = 255
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 4
VAL_SPLIT = 0.1

# Baseline weighted-loss setup used during training in notebook 03.
BASE_CLASS_WEIGHTS = torch.tensor([1.0, 2.5, 1.5, 4.0], dtype=torch.float32)

# Toggle to test data-driven class weights from the train split stats below.
USE_DYNAMIC_CLASS_WEIGHTS = True
CLASS_WEIGHTS = BASE_CLASS_WEIGHTS  # backward-compatible alias

device = get_device()

def build_unet(encoder_name: str, num_classes: int, device: torch.device):
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights="imagenet",
        in_channels=3,
        classes=num_classes,
    ).to(device)

def infer_encoder_name(state_dict: dict) -> str:
    # Checkpoint from 03 can be either resnet34 (CUDA path) or mobilenet_v2 (CPU path).
    keys = list(state_dict.keys())
    if any(k.startswith("encoder.layer1.") for k in keys):
        return "resnet34"
    if any(k.startswith("encoder.features.") for k in keys):
        return "mobilenet_v2"
    raise RuntimeError("Could not infer encoder type from checkpoint state_dict.")

Using device: cuda


## Step 2 — Load Checkpoint

In [4]:
checkpoint_path = MODELS_DIR / "weighted_unet_epoch03.pth"

if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    encoder_name = infer_encoder_name(checkpoint["model_state_dict"])

    model = build_unet(encoder_name, NUM_CLASSES, device)
    optimizer = torch.optim.Adam(model.parameters())  # needed by load_checkpoint

    epoch = load_checkpoint(model, optimizer, checkpoint_path, device)
    print(f"Loaded checkpoint from epoch {epoch}")
    print(f"Loaded model architecture: U-Net with encoder='{encoder_name}'")
else:
    raise FileNotFoundError(
        f"Checkpoint not found at {checkpoint_path}. "
        "Run 03_baseline_training.ipynb first."
    )

Checkpoint loaded from C:\Users\Jacob\AI4Mars\models\weighted_unet_epoch03.pth  (epoch 3)
Loaded checkpoint from epoch 3
Loaded model architecture: U-Net with encoder='resnet34'


## Step 3 — Recreate the Validation Set

## Step 3.5 — Diagnose Class Imbalance and Propose Weights

In [6]:
DATASET_ROOT = detect_dataset_root(RAW_DATA_DIR)
MANIFEST_DIR = ARTIFACTS_DIR / "manifests"
SPLIT_DIR = MANIFEST_DIR / "splits"
DATASET_MANIFEST = MANIFEST_DIR / "ai4mars_dataset_manifest.csv"

if not DATASET_MANIFEST.exists():
    manifest_rows = write_dataset_manifest(DATASET_ROOT, DATASET_MANIFEST)
    print(f"Wrote dataset manifest: {DATASET_MANIFEST} ({len(manifest_rows)} rows)")
else:
    print(f"Using existing dataset manifest: {DATASET_MANIFEST}")

expected_split_paths = {
    "train": SPLIT_DIR / "train_nav.csv",
    "val": SPLIT_DIR / "val_nav.csv",
    "test_min1_100agree": SPLIT_DIR / "test_min1_100agree_nav.csv",
    "test_min2_100agree": SPLIT_DIR / "test_min2_100agree_nav.csv",
    "test_min3_100agree": SPLIT_DIR / "test_min3_100agree_nav.csv",
}

if not all(path.exists() for path in expected_split_paths.values()):
    split_manifest_paths = build_split_manifests(
        DATASET_MANIFEST,
        SPLIT_DIR,
        train_ratio=0.80,
        seed=42,
        label_scheme="NAV",
    )
    print(f"Generated split manifests under {SPLIT_DIR}")
else:
    split_manifest_paths = expected_split_paths

train_manifest = split_manifest_paths["train"]
val_manifest = split_manifest_paths["val"]
test_manifest_paths = {
    key: value for key, value in split_manifest_paths.items() if key.startswith("test_")
}

train_pairs = load_pairs_from_manifest(
    train_manifest,
    dataset_root=DATASET_ROOT,
    required_label_scheme="NAV",
    require_shape_match=True,
)
val_pairs = load_pairs_from_manifest(
    val_manifest,
    dataset_root=DATASET_ROOT,
    required_label_scheme="NAV",
    require_shape_match=True,
)
expert_test_pairs = {
    name: load_pairs_from_manifest(
        path,
        dataset_root=DATASET_ROOT,
        required_label_scheme="NAV",
        require_shape_match=True,
    )
    for name, path in sorted(test_manifest_paths.items())
}
test_pairs = expert_test_pairs["test_min3_100agree"]

train_dataset = AI4MarsDataset(train_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
val_dataset = AI4MarsDataset(val_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)
test_dataset = AI4MarsDataset(test_pairs, image_size=IMAGE_SIZE, require_original_shape_match=True)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train samples (manifest split): {len(train_dataset)}")
print(f"Validation samples (manifest split): {len(val_dataset)}")
print(f"Test samples (min3 expert gold): {len(test_dataset)}")

train_loader_stats = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
class_pixel_counts = torch.zeros(NUM_CLASSES, dtype=torch.long)
for _, masks in train_loader_stats:
    valid = masks != IGNORE_INDEX
    for c in range(NUM_CLASSES):
        class_pixel_counts[c] += ((masks == c) & valid).sum()

total_labeled_pixels = class_pixel_counts.sum().item()
class_freq = class_pixel_counts.float() / max(total_labeled_pixels, 1)
inverse_freq = 1.0 / torch.clamp(class_freq, min=1e-8)
dynamic_class_weights = inverse_freq / inverse_freq.mean()

print("Train-split labeled pixel counts by class:")
for c in range(NUM_CLASSES):
    name = CLASS_NAMES.get(c, f"class_{c}")
    print(
        f"  class {c} ({name:>10s}): {class_pixel_counts[c].item():>12d} pixels "
        f"({class_freq[c].item() * 100:6.2f}%)"
    )

print("\nBaseline class weights (from notebook 03):")
for c in range(NUM_CLASSES):
    name = CLASS_NAMES.get(c, f"class_{c}")
    print(f"  class {c} ({name:>10s}): {BASE_CLASS_WEIGHTS[c].item():.4f}")

print("\nProposed dynamic class weights (inverse-frequency, mean-normalized):")
for c in range(NUM_CLASSES):
    name = CLASS_NAMES.get(c, f"class_{c}")
    print(f"  class {c} ({name:>10s}): {dynamic_class_weights[c].item():.4f}")

ACTIVE_CLASS_WEIGHTS = dynamic_class_weights if USE_DYNAMIC_CLASS_WEIGHTS else BASE_CLASS_WEIGHTS
print(f"\nUsing {'dynamic' if USE_DYNAMIC_CLASS_WEIGHTS else 'baseline'} class weights for loss.")
print(f"Active weights: {ACTIVE_CLASS_WEIGHTS.tolist()}")

expected_checkpoint_metadata = build_checkpoint_metadata(
    project_root=PROJECT_ROOT,
    dataset_manifest_path=DATASET_MANIFEST,
    split_manifest_paths=split_manifest_paths,
    active_split_name="val",
    preprocessing={
        "image_size": list(IMAGE_SIZE),
        "require_original_shape_match": True,
        "label_scheme": "NAV",
    },
    loss_name="CrossEntropyLoss",
    loss_weights=[float(value) for value in ACTIVE_CLASS_WEIGHTS.tolist()],
    model_name=f"Unet/{encoder_name}",
    seed=42,
)

checkpoint_metadata = checkpoint.get("metadata") if "checkpoint" in globals() else None
if checkpoint_metadata is None:
    raise RuntimeError("Checkpoint metadata missing; strict checkpoint/split compatibility is required.")

mismatches = []
for key, expected_value in expected_checkpoint_metadata.items():
    actual_value = checkpoint_metadata.get(key)
    if actual_value != expected_value:
        mismatches.append((key, expected_value, actual_value))

if mismatches:
    mismatch_text = "; ".join(
        f"{key}: expected={expected!r}, actual={actual!r}"
        for key, expected, actual in mismatches
    )
    raise RuntimeError(f"Checkpoint metadata mismatch: {mismatch_text}")

print("Checkpoint metadata matches dataset manifest and split manifests.")

val_source_records = []
for sample_idx, (image_path, mask_path) in enumerate(val_dataset.pairs):
    val_source_records.append(
        {
            "sample_idx": sample_idx,
            "split": "val",
            "image_path": str(image_path),
            "mask_path": str(mask_path),
            "image_file_sha256": sha256_file(Path(image_path)),
            "mask_file_sha256": sha256_file(Path(mask_path)),
            "source_stem": Path(image_path).stem,
        }
    )
print(f"Prepared provenance records for {len(val_source_records)} validation samples.")


Using existing dataset manifest: C:\Users\Jacob\AI4Mars\artifacts\manifests\ai4mars_dataset_manifest.csv
Train samples (manifest split): 28487
Validation samples (manifest split): 7038
Test samples (min3 expert gold): 526
Train-split labeled pixel counts by class:
  class 0 (      soil):    485292480 pixels ( 39.57%)
  class 1 (   bedrock):    348943148 pixels ( 28.45%)
  class 2 (      sand):    336692129 pixels ( 27.45%)
  class 3 (  big_rock):     55489016 pixels (  4.52%)

Baseline class weights (from notebook 03):
  class 0 (      soil): 1.0000
  class 1 (   bedrock): 2.5000
  class 2 (      sand): 1.5000
  class 3 (  big_rock): 4.0000

Proposed dynamic class weights (inverse-frequency, mean-normalized):
  class 0 (      soil): 0.3180
  class 1 (   bedrock): 0.4423
  class 2 (      sand): 0.4584
  class 3 (  big_rock): 2.7813

Using dynamic class weights for loss.
Active weights: [0.3180195689201355, 0.4422855079174042, 0.4583786725997925, 2.7813162803649902]
Checkpoint metadata m

## Step 4 — Run Predictions and Compute Metrics

In [7]:
# Safety first: free any cached CUDA memory from prior runs.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

weights_for_loss = ACTIVE_CLASS_WEIGHTS if 'ACTIVE_CLASS_WEIGHTS' in globals() else BASE_CLASS_WEIGHTS
loss_fn = nn.CrossEntropyLoss(weight=weights_for_loss.to(device), ignore_index=IGNORE_INDEX)
print(f"Loss weights in use: {weights_for_loss.tolist()}")

# 1) Smoke-test one batch to validate memory behavior before full eval.
model.eval()
with torch.no_grad():
    images, masks = next(iter(val_loader))
    images = images.to(device)
    outputs = model(images)

    # Move results back to CPU before further analysis/visualization.
    preds = outputs.argmax(dim=1).cpu()
    images = images.cpu()
    masks = masks.cpu()

print(f"Smoke test batch ok: images={tuple(images.shape)}, preds={tuple(preds.shape)}")

# 2) Optional full validation pass. Keep False while debugging OOM.
RUN_FULL_EVAL = True
RANKED_SAMPLE_COUNT = 3

# Ranking filters for informative overlays (avoid trivial all-soil and mostly-ignore cases).
MIN_VALID_PIXELS_FOR_RANKING = 4096
MAX_IGNORED_FRACTION_FOR_RANKING = 0.60
MIN_GT_CLASSES_FOR_RANKING = 2

if RUN_FULL_EVAL:
    all_val_indices = []
    total_loss = 0.0
    finite_loss_batches = 0
    skipped_all_ignore_loss_batches = 0
    total_correct_pixels = 0
    total_valid_pixels = 0
    class_intersections = torch.zeros(NUM_CLASSES, dtype=torch.long)
    class_unions = torch.zeros(NUM_CLASSES, dtype=torch.long)
    confusion_matrix = torch.zeros((NUM_CLASSES, NUM_CLASSES), dtype=torch.long)
    total_ignored_pixels = 0
    total_pixels = 0
    sample_miou_sum = 0.0
    sample_miou_count = 0
    sample_eval_rows = []
    val_audit_rows = []

    with torch.no_grad():
        for batch_idx, (images, masks) in enumerate(val_loader):
            batch_start_idx = batch_idx * BATCH_SIZE
            batch_size_actual = masks.shape[0]

            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)  # [B, num_classes, H, W]
            valid_batch = masks != IGNORE_INDEX
            if valid_batch.any():
                loss = loss_fn(logits, masks)
                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite validation loss in batch {batch_idx} with valid target pixels.")
                total_loss += loss.item()
                finite_loss_batches += 1
            else:
                skipped_all_ignore_loss_batches += 1

            pred_batch = logits.argmax(dim=1)  # [B, H, W]
            total_correct_pixels += ((pred_batch == masks) & valid_batch).sum().item()
            total_valid_pixels += valid_batch.sum().item()
            total_ignored_pixels += (masks == IGNORE_INDEX).sum().item()
            total_pixels += masks.numel()
            confusion_matrix += segmentation_confusion_matrix(
                pred_batch,
                masks,
                num_classes=NUM_CLASSES,
                ignore_index=IGNORE_INDEX,
            ).cpu()

            for c in range(NUM_CLASSES):
                pred_c = (pred_batch == c) & valid_batch
                true_c = (masks == c) & valid_batch
                class_intersections[c] += (pred_c & true_c).sum().cpu()
                class_unions[c] += (pred_c | true_c).sum().cpu()

            pred_batch_cpu = pred_batch.cpu()
            masks_cpu = masks.cpu()
            all_val_indices.extend(range(batch_start_idx, batch_start_idx + batch_size_actual))

            for sample_offset in range(batch_size_actual):
                sample_idx = batch_start_idx + sample_offset
                pred_sample = pred_batch_cpu[sample_offset]
                target_sample = masks_cpu[sample_offset]
                valid_mask = target_sample != IGNORE_INDEX
                valid_pixels = int(valid_mask.sum().item())
                total_pixels = int(target_sample.numel())
                ignored_fraction = float((target_sample == IGNORE_INDEX).sum().item() / max(total_pixels, 1))

                sample_iou = intersection_over_union(
                    pred_sample.unsqueeze(0),
                    target_sample.unsqueeze(0),
                    num_classes=NUM_CLASSES,
                    ignore_index=IGNORE_INDEX,
                )
                valid_scores = [iou for iou in sample_iou if iou is not None]
                sample_miou = (sum(valid_scores) / len(valid_scores)) if valid_scores else None
                if sample_miou is not None:
                    sample_miou_sum += sample_miou
                    sample_miou_count += 1

                unique_gt_classes = sorted(torch.unique(target_sample).tolist())
                unique_pred_classes = sorted(torch.unique(pred_sample).tolist())
                distinct_gt_non_ignore = sorted(c for c in unique_gt_classes if c != IGNORE_INDEX)

                sample_row = {
                    "sample_idx": sample_idx,
                    "unique_gt_classes": unique_gt_classes,
                    "unique_pred_classes": unique_pred_classes,
                    "valid_pixels": valid_pixels,
                    "ignored_fraction": ignored_fraction,
                    "per_class_iou": sample_iou,
                    "valid_class_count": len(valid_scores),
                    "sample_miou": sample_miou,
                    "distinct_gt_non_ignore_count": len(distinct_gt_non_ignore),
                }
                sample_eval_rows.append(sample_row)

                source_record = val_source_records[sample_idx]
                audit_row = dict(source_record)
                audit_row.update(
                    {
                        "target_tensor_sha256": sha256_tensor(target_sample),
                        "pred_tensor_sha256": sha256_tensor(pred_sample),
                        "recorded_miou": sample_miou,
                        "unique_gt_classes": sample_row["unique_gt_classes"],
                        "unique_pred_classes": sample_row["unique_pred_classes"],
                        "valid_pixels": valid_pixels,
                        "ignored_fraction": ignored_fraction,
                        "per_class_iou": sample_iou,
                        "valid_class_count": len(valid_scores),
                    }
                )
                val_audit_rows.append(audit_row)

            del logits, pred_batch, pred_batch_cpu, masks_cpu, valid_batch

    if len(all_val_indices) != len(val_dataset):
        raise RuntimeError("Validation index tracking length mismatch.")
    if all_val_indices != list(range(len(val_dataset))):
        raise RuntimeError("Validation loader order mismatch; expected contiguous dataset indices.")

    eligible_sample_ids = [
        row["sample_idx"]
        for row in sample_eval_rows
        if row["valid_pixels"] > 0 and row["valid_class_count"] > 0 and row["sample_miou"] is not None
    ]
    if not eligible_sample_ids:
        raise RuntimeError("No eligible validation samples with non-empty valid labels.")

    informative_sample_ids = [
        row["sample_idx"]
        for row in sample_eval_rows
        if row["sample_idx"] in eligible_sample_ids
        and row["valid_pixels"] >= MIN_VALID_PIXELS_FOR_RANKING
        and row["ignored_fraction"] <= MAX_IGNORED_FRACTION_FOR_RANKING
        and row["distinct_gt_non_ignore_count"] >= MIN_GT_CLASSES_FOR_RANKING
    ]

    excluded_count = len(sample_eval_rows) - len(eligible_sample_ids)
    print(
        f"Ranking eligibility: {len(eligible_sample_ids)} included, {excluded_count} excluded (empty valid mask or no valid IoU classes)."
    )

    print(
        "Informative ranking filter -> "
        f"valid_pixels>={MIN_VALID_PIXELS_FOR_RANKING}, "
        f"ignored_fraction<={MAX_IGNORED_FRACTION_FOR_RANKING:.2f}, "
        f"distinct_gt_non_ignore>={MIN_GT_CLASSES_FOR_RANKING}."
    )
    print(f"Informative candidates: {len(informative_sample_ids)}")

    used_informative_ranking = len(informative_sample_ids) >= RANKED_SAMPLE_COUNT * 2
    ranking_pool = informative_sample_ids if used_informative_ranking else eligible_sample_ids
    if ranking_pool is eligible_sample_ids:
        print("Warning: informative pool too small; falling back to all eligible samples for ranking.")

    ranked_samples = sorted(
        ranking_pool,
        key=lambda idx: sample_eval_rows[idx]["sample_miou"],
        reverse=True,
    )
    best_sample_ids = ranked_samples[:RANKED_SAMPLE_COUNT]
    worst_sample_ids = list(reversed(ranked_samples[-RANKED_SAMPLE_COUNT:]))

    mean_val_loss = (total_loss / finite_loss_batches) if finite_loss_batches > 0 else float("nan")
    pix_acc = total_correct_pixels / max(total_valid_pixels, 1)
    per_class_iou = [
        (class_intersections[c].item() / class_unions[c].item()) if class_unions[c].item() > 0 else None
        for c in range(NUM_CLASSES)
    ]
    valid_global_ious = [iou for iou in per_class_iou if iou is not None]
    m_iou = (sum(valid_global_ious) / len(valid_global_ious)) if valid_global_ious else 0.0
    macro_sample_miou = (sample_miou_sum / sample_miou_count) if sample_miou_count else None

    if int(confusion_matrix.sum().item()) != total_valid_pixels:
        raise RuntimeError("Confusion matrix total does not match the valid-pixel count.")
    if not torch.equal(confusion_matrix.diag(), class_intersections):
        raise RuntimeError("Confusion matrix diagonal does not match per-class intersections.")

    class_support = confusion_matrix.sum(dim=1)
    predicted_pixel_counts = confusion_matrix.sum(dim=0)
    class_report = []
    for c in range(NUM_CLASSES):
        true_positive = int(confusion_matrix[c, c].item())
        support = int(class_support[c].item())
        predicted_count = int(predicted_pixel_counts[c].item())
        false_positive = predicted_count - true_positive
        false_negative = support - true_positive
        precision = true_positive / predicted_count if predicted_count else None
        recall = true_positive / support if support else None
        class_report.append(
            {
                "class_id": c,
                "class_name": CLASS_NAMES.get(c, f"class_{c}"),
                "support_pixels": support,
                "predicted_pixels": predicted_count,
                "true_positive": true_positive,
                "false_positive": false_positive,
                "false_negative": false_negative,
                "iou": per_class_iou[c],
                "precision": precision,
                "recall": recall,
            }
        )

    evaluation_summary = {
        "split": "val",
        "checkpoint_path": str(checkpoint_path),
        "checkpoint_epoch": int(epoch),
        "checkpoint_metadata": checkpoint_metadata,
        "global_metrics": {
            "weighted_validation_loss": mean_val_loss if finite_loss_batches else None,
            "pixel_accuracy": pix_acc,
            "global_mean_iou": m_iou,
            "macro_sample_mean_iou": macro_sample_miou,
        },
        "pixel_accounting": {
            "total_pixels": total_pixels,
            "valid_pixels": total_valid_pixels,
            "ignored_pixels": total_ignored_pixels,
            "ignored_fraction": total_ignored_pixels / max(total_pixels, 1),
        },
        "loss_batches": {
            "finite": finite_loss_batches,
            "skipped_all_ignore": skipped_all_ignore_loss_batches,
        },
        "confusion_matrix_rows_ground_truth_columns_prediction": confusion_matrix.tolist(),
        "per_class": class_report,
        "ranking": {
            "eligible_samples": len(eligible_sample_ids),
            "informative_samples": len(informative_sample_ids),
            "ranking_pool_samples": len(ranking_pool),
            "used_informative_filter": used_informative_ranking,
            "fallback_to_all_eligible": not used_informative_ranking,
        },
    }

    print(f"Finite loss batches        : {finite_loss_batches}")
    print(f"Skipped all-ignore batches : {skipped_all_ignore_loss_batches}")
    print(f"Validation loss (weighted) : {mean_val_loss:.4f}")
    print(f"Pixel accuracy             : {pix_acc:.4f}")
    print(f"Mean IoU                   : {m_iou:.4f}")
    print(f"Macro sample mean IoU      : {macro_sample_miou:.4f}" if macro_sample_miou is not None else "Macro sample mean IoU      : N/A")
    print(f"Ignored pixel fraction      : {evaluation_summary['pixel_accounting']['ignored_fraction']:.4f}")
    print("\nPer-class support, IoU, precision, and recall:")
    for row in class_report:
        iou_str = f"{row['iou']:.4f}" if row['iou'] is not None else "N/A"
        precision_str = f"{row['precision']:.4f}" if row['precision'] is not None else "N/A"
        recall_str = f"{row['recall']:.4f}" if row['recall'] is not None else "N/A"
        print(f"  {row['class_name']:>10s}: support={row['support_pixels']:>10d}, predicted={row['predicted_pixels']:>10d}, IoU={iou_str}, precision={precision_str}, recall={recall_str}")
    print("\nPer-class IoU:")
    for c, iou in enumerate(per_class_iou):
        name = CLASS_NAMES.get(c, f"class_{c}")
        iou_str = f"{iou:.4f}" if iou is not None else "N/A (not in batch)"
        print(f"  class {c} ({name:>10s}): {iou_str}")

    print("\nBest validation samples by mean IoU:")
    for sample_id in best_sample_ids:
        row = sample_eval_rows[sample_id]
        sample_score = row["sample_miou"]
        print(
            f"  sample {sample_id + 1:>4d}: mIoU={sample_score:.4f}, "
            f"valid_pixels={row['valid_pixels']}, ignored_fraction={row['ignored_fraction']:.4f}, "
            f"gt_classes={row['distinct_gt_non_ignore_count']}"
        )

    print("\nWorst validation samples by mean IoU:")
    for sample_id in worst_sample_ids:
        row = sample_eval_rows[sample_id]
        sample_score = row["sample_miou"]
        print(
            f"  sample {sample_id + 1:>4d}: mIoU={sample_score:.4f}, "
            f"valid_pixels={row['valid_pixels']}, ignored_fraction={row['ignored_fraction']:.4f}, "
            f"gt_classes={row['distinct_gt_non_ignore_count']}"
        )
else:
    print("Skipped full evaluation (RUN_FULL_EVAL=False). Set True after smoke test passes.")


Loss weights in use: [0.3180195689201355, 0.4422855079174042, 0.4583786725997925, 2.7813162803649902]
Smoke test batch ok: images=(4, 3, 256, 256), preds=(4, 256, 256)
Ranking eligibility: 6862 included, 176 excluded (empty valid mask or no valid IoU classes).
Informative ranking filter -> valid_pixels>=4096, ignored_fraction<=0.60, distinct_gt_non_ignore>=2.
Informative candidates: 3427
Finite loss batches        : 1735
Skipped all-ignore batches : 25
Validation loss (weighted) : 0.8180
Pixel accuracy             : 0.6802
Mean IoU                   : 0.4618
Macro sample mean IoU      : 0.3207
Ignored pixel fraction      : 2410.9718

Per-class support, IoU, precision, and recall:
        soil: support= 116103845, predicted=  93189280, IoU=0.4961, precision=0.7447, recall=0.5977
     bedrock: support=  91775250, predicted=  78549321, IoU=0.6179, precision=0.8282, recall=0.7088
        sand: support=  82329612, predicted= 114398535, IoU=0.5078, precision=0.5792, recall=0.8048
    big_roc

## Step 5 — Visual Prediction Comparison

Show the best and worst validation examples, and save overlay figures for later review.

In [8]:
analysis_dir = FIGURES_DIR / "04_evaluation_error_analysis"
analysis_dir.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
DISPLAY_INLINE_FIGURES = False

if RUN_FULL_EVAL:
    audit_dir = PREDICTIONS_DIR / "audit"
    audit_dir.mkdir(parents=True, exist_ok=True)

    evaluation_summary_path = audit_dir / "val_evaluation_summary.json"
    with open(evaluation_summary_path, "w", encoding="utf-8") as f:
        json.dump(evaluation_summary, f, indent=2)
    print(f"Saved validation evaluation summary to {evaluation_summary_path}")

    full_audit_jsonl = audit_dir / "val_sample_provenance.jsonl"
    with open(full_audit_jsonl, "w", encoding="utf-8") as f:
        for row in val_audit_rows:
            f.write(json.dumps(row) + "\n")
    print(f"Saved full validation provenance to {full_audit_jsonl}")

    plotted_audit_rows = []
    example_groups = [
        ("best", best_sample_ids),
        ("worst", worst_sample_ids),
    ]

    for group_name, sample_ids in example_groups:
        for rank, sample_id in enumerate(sample_ids, start=1):
            image_tensor, target_mask = val_dataset[sample_id]
            with torch.no_grad():
                pred_mask = model(image_tensor.unsqueeze(0).to(device)).argmax(dim=1).squeeze(0).cpu()

            expected = val_audit_rows[sample_id]
            target_hash_now = sha256_tensor(target_mask)
            pred_hash_now = sha256_tensor(pred_mask)
            if target_hash_now != expected["target_tensor_sha256"]:
                raise RuntimeError(f"Target mask hash mismatch at sample {sample_id}")
            if pred_hash_now != expected["pred_tensor_sha256"]:
                raise RuntimeError(f"Prediction hash mismatch at sample {sample_id}")

            sample_iou_now = intersection_over_union(
                pred_mask.unsqueeze(0),
                target_mask.unsqueeze(0),
                num_classes=NUM_CLASSES,
                ignore_index=IGNORE_INDEX,
            )
            valid_scores_now = [iou for iou in sample_iou_now if iou is not None]
            sample_miou_now = (sum(valid_scores_now) / len(valid_scores_now)) if valid_scores_now else None
            recorded_miou = expected.get("recorded_miou")

            if sample_miou_now is None or recorded_miou is None:
                raise RuntimeError(f"Expected non-empty valid IoU classes for plotted sample {sample_id}.")
            if abs(sample_miou_now - recorded_miou) > 1e-8:
                raise RuntimeError(
                    f"Recorded mIoU mismatch at sample {sample_id}: recorded={recorded_miou}, recomputed={sample_miou_now}"
                )

            image_np = image_tensor.permute(1, 2, 0).numpy()
            target_np = target_mask.numpy()
            pred_np = pred_mask.numpy()

            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
            show_image(image_np, title="Image", ax=axes[0])
            show_image_mask_overlay(image_np, target_np, title="Ground Truth Overlay", ax=axes[1])
            show_image_mask_overlay(image_np, pred_np, title="Prediction Overlay", ax=axes[2])
            plt.suptitle(
                f"{group_name.title()} sample {rank} (validation idx {sample_id + 1}, mIoU={sample_miou_now:.4f})",
                fontsize=11,
            )
            plt.tight_layout()

            overlay_path = analysis_dir / f"{group_name}_sample_{rank:02d}_idx_{sample_id + 1:04d}_miou_{sample_miou_now:.4f}.png"
            fig.savefig(overlay_path, dpi=150, bbox_inches="tight")
            print(f"Saved {group_name} overlay to {overlay_path}")

            print("  Dataset index:", sample_id)
            print("  Image filename:", Path(expected["image_path"]).name)
            print("  Mask filename:", Path(expected["mask_path"]).name)
            print("  Unique GT classes:", expected["unique_gt_classes"])
            print("  Unique predicted classes:", expected["unique_pred_classes"])
            print("  Number of valid pixels:", expected["valid_pixels"])
            print("  Fraction of ignored pixels:", f"{expected['ignored_fraction']:.6f}")
            print("  Per-class IoU:", expected["per_class_iou"])
            print("  Mean IoU:", f"{recorded_miou:.6f}")

            if DISPLAY_INLINE_FIGURES:
                plt.show()
            plt.close(fig)

            plotted_row = dict(expected)
            plotted_row.update(
                {
                    "group": group_name,
                    "rank": rank,
                    "miou": float(sample_miou_now),
                    "overlay_path": str(overlay_path),
                }
            )
            plotted_audit_rows.append(plotted_row)

    plotted_audit_csv = audit_dir / "plotted_sample_provenance.csv"
    with open(plotted_audit_csv, "w", newline="", encoding="utf-8") as f:
        fieldnames = [
            "group", "rank", "sample_idx", "miou", "split", "source_stem",
            "image_path", "mask_path", "image_file_sha256", "mask_file_sha256",
            "target_tensor_sha256", "pred_tensor_sha256", "recorded_miou",
            "valid_pixels", "ignored_fraction", "unique_gt_classes", "unique_pred_classes",
            "valid_class_count",
            "per_class_iou", "overlay_path",
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(plotted_audit_rows)
    print(f"Saved plotted-sample provenance table to {plotted_audit_csv}")
else:
    print("Run the full evaluation first to generate best/worst overlays.")


Saved validation evaluation summary to C:\Users\Jacob\AI4Mars\outputs\predictions\audit\val_evaluation_summary.json
Saved full validation provenance to C:\Users\Jacob\AI4Mars\outputs\predictions\audit\val_sample_provenance.jsonl
Saved best overlay to C:\Users\Jacob\AI4Mars\outputs\figures\04_evaluation_error_analysis\best_sample_01_idx_2160_miou_0.9999.png
  Dataset index: 2159
  Image filename: 1n235691806eff85mep1981l0m1.JPG
  Mask filename: 1n235691806eff85mep1981l0m1_merged6.png
  Unique GT classes: [1, 2, 255]
  Unique predicted classes: [0, 1, 2, 3]
  Number of valid pixels: 29510
  Fraction of ignored pixels: 0.549713
  Per-class IoU: [None, 0.9999607673898544, 0.9997513674788663, None]
  Mean IoU: 0.999856
Saved best overlay to C:\Users\Jacob\AI4Mars\outputs\figures\04_evaluation_error_analysis\best_sample_02_idx_7007_miou_0.9865.png
  Dataset index: 7006
  Image filename: NLB_529421365EDR_F0581836CCAM15903M1.JPG
  Mask filename: NLB_529421365EDR_F0581836CCAM15903M1.png
  Uniqu

In [ ]:
#confusion matrix visualization
#rows = ground truth, columns = predictions

class_labels = [
    CLASS_NAMES.get(class_id, f"class_{class_id}")
    for class_id in range(NUM_CLASSES)
]

cm_counts = confusion_matrix.cpu().numpy()

#normalize ground truth row independently to get per-class accuracy
row_totals = cm_counts.sum(axis=1, keepdims=True)

cm_normalized = cm_counts.astype(float)
cm_normalized = cm_normalized / row_totals.clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

#raw-count confusion matrix

count_image = axes[0].imshow(cm_counts)

axes[0].set_title("Confusion Matrix (raw counts)")
axes[0].set_xlabel("Predicted Class")
axes[0].set_ylabel("Ground Truth Class")

axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_xticklabels(class_labels, rotation=45, ha="right")
axes[0].set_yticklabels(class_labels)

for true_class in range(NUM_CLASSES):
    for pred_class in range(NUM_CLASSES):
        axes[0].text(
            pred_class,
            true_class,
            f"{cm_counts[true_class, pred_class]:,}",
            ha="center",
            va="center",
        )

fig.colorbar(count_image, ax=axes[0], label="pixel count")

#row normalized confusion matrix

normalized_image = axes[1].imshow(
    cm_normalized,
    vmin=0.0,
    vmax=1.0,
)

axes[1].set_title("Validation Confusion Matrix — Row Normalized")
axes[1].set_xlabel("Predicted class")
axes[1].set_ylabel("Ground-truth class")

axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_yticks(range(NUM_CLASSES))
axes[1].set_xticklabels(class_labels, rotation=45, ha="right")
axes[1].set_yticklabels(class_labels)

for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        axes[1].text(
            predicted_class,
            true_class,
            f"{cm_normalized[true_class, predicted_class]:.1%}",
            ha="center",
            va="center",
        )

fig.colorbar(
    normalized_image,
    ax=axes[1],
    label="Fraction of true-class pixels",
)

fig.suptitle(
    "AI4Mars Validation Confusion Matrices\n"
    "Rows are ground truth; columns are model predictions"
)

fig.tight_layout()

confusion_matrix_path = (
    FIGURES_DIR / "validation_confusion_matrix.png"
)

fig.savefig(
    confusion_matrix_path,
    dpi=300,
    bbox_inches="tight",
)

print(f"Saved confusion matrix figure to: {confusion_matrix_path}")

plt.show()        



---

### Observations

| # | Failure type | Example samples | Likely cause |
|---|---|---|---|
| 1 | Sample 1: label under-coverage vs visible rocks | Sample 1 | GT marks almost everything as soil/sand while image contains many rock-like structures; weak/incomplete annotation inflates apparent model error |
| 2 | Sample 1: big_rock over-prediction | Sample 1 | Model creates large big_rock islands not supported by GT mask; big_rock precision is low under class imbalance and texture ambiguity |
| 3 | Sample 1: scattered bedrock speckles | Sample 1 | Small isolated bedrock patches suggest noisy logits and insufficient spatial regularization |
| 4 | Sample 2: strong domain shift (wheel/hardware scene) | Sample 2 | Training data appears terrain-centric; rover hardware textures trigger large false positive big_rock regions |
| 5 | Sample 2: sparse GT polygon vs dense prediction | Sample 2 | GT has a small sand wedge only, while model predicts many classes; annotation sparsity/coverage mismatch dominates perceived failure |
| 6 | Sample 2: fragmented class islands | Sample 2 | Pepper-like bedrock/rock blobs indicate unstable boundaries and over-segmentation in cluttered scenes |
| 7 | Sample 3: ignore-region leakage | Sample 3 | GT includes large ignore zones, but prediction assigns valid classes inside/near those regions; ignore handling is weak at boundaries |
| 8 | Sample 3: false bedrock blob in mostly-soil frame | Sample 3 | Isolated bedrock prediction appears without clear visual support; likely confidence calibration issue |
| 9 | Sample 4: missed tiny big_rock target (false negative) | Sample 4 | Small labeled rock is not recovered; downsampling and low effective resolution hurt small-object recall |
| 10 | Sample 4: horizon band class confusion | Sample 4 | Predicted upper band as sand while GT top is ignore/background; model overextends terrain classes into non-target area |
| 11 | Sample 4: overly smooth segmentation field | Sample 4 | Prediction collapses to broad horizontal regions rather than object-aligned boundaries |
| 12 | Sample 5: right-edge bedrock hallucination | Sample 5 | Large bedrock region appears on right border without GT support; edge artifacts and contrast bias likely contributors |
| 13 | Sample 5: ignore-dominant GT mismatch | Sample 5 | GT has substantial ignore masks, but model fills most of frame with valid classes, reducing alignment with labels |
| 14 | Sample 6: severe class disagreement in mixed rover/terrain scene | Sample 6 | GT marks large bedrock+ignore structures while prediction redistributes area across soil/sand/big_rock; possible annotation semantics mismatch |
| 15 | Sample 6: large contiguous big_rock false positives | Sample 6 | Bottom-half prediction contains oversized big_rock regions not reflected in GT, indicating low big_rock precision |
| 16 | Sample 6: bedrock under-detection | Sample 6 | GT bedrock occupies wide regions, but prediction yields only small bedrock patches |
| 17 | Sample 7: bedrock under-segmentation on slab field | Sample 7 | Many slab-like bedrock GT regions are reassigned to sand/soil, lowering bedrock recall |
| 18 | Sample 7: big_rock over-expansion | Sample 7 | Predicted big_rock spreads beyond labeled area and merges with neighboring regions |
| 19 | Sample 7: boundary bleeding between classes | Sample 7 | Adjacent classes merge with soft, shifted borders instead of matching sharp GT transitions |
| 20 | Sample 8: ignore-band mismatch and over-classification | Sample 8 | GT is largely ignore with narrow labeled bands, but prediction covers frame with soil/sand/bedrock labels |
| 21 | Sample 8: low-light texture confusion | Sample 8 | Dark dune textures are mapped to mixed classes with little structural evidence, suggesting weak photometric robustness |
| 22 | Cross-sample: border/corner artifacts | Samples 1, 2, 5, 8 | Frequent false regions near edges imply padding/resize artifacts and insufficient border-aware augmentation |
| 23 | Cross-sample: ignore-boundary instability | Samples 3, 4, 5, 6, 8 | Hard transitions around ignore masks are not preserved; loss does not strongly constrain near-ignore uncertainty |
| 24 | Cross-sample: small-object recall deficit | Samples 4, 7 | Small/compact rocks are often missed or absorbed into soil/sand due to scale and feature resolution limits |
| 25 | Cross-sample: class imbalance effects on big_rock | Samples 1, 2, 6, 7 | Big_rock shows both false positives and misses, consistent with unstable decision boundary for minority class |

### Ideas for Improvement

- [ ] Add stronger augmentation (flip, color jitter, contrast).
- [ ] Tune class weights using measured label frequencies from NAV masks.
- [ ] Train longer and use LR scheduling (e.g., cosine/one-cycle).
- [ ] Try larger input size for small rock recall if VRAM allows.
- [ ] Add overlay-only audits focused on the hardest IoU class.